<a href="https://colab.research.google.com/github/Tarooba05/stock-market-analytics/blob/main/Week-1/homework1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [67]:
# install the main library YFinance
!pip install yfinance

In [68]:
# IMPORTS
import numpy as np
import pandas as pd

#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date


# **Homework**

In [150]:
#Question 1
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}
tables = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", storage_options=headers)
# find the "changes" table (has Date, Added, Removed columns)
for i, t in enumerate(tables):
    print(i, t.columns.tolist()[:6])

0 ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'Headquarters Location', 'Date added']
1 ['vteS&P 500 companies', 'vteS&P 500 companies.1']


In [154]:
changes = tables[0]
changes.columns = [col.replace(' ', '_') for col in changes.columns]
date_col = 'Date_added'
changes[date_col] = pd.to_datetime(changes[date_col], errors='coerce')
recent = changes[changes[date_col] >= '2020-01-01'].copy()
recent['year'] = recent[date_col].dt.year
recent['year'].value_counts().sort_index()

,count
year,
2020,10
2021,10
2022,15
2023,15
2024,16
2025,18
2026,13


In [152]:
#Question 2
indexes = {
    'S&P500 (US)': '^GSPC',
    'Shanghai (China)': '000001.SS',
    'Hang Seng (HK)': '^HSI',
    'ASX 200 (Australia)': '^AXJO',
    'Nifty 50 (India)': '^NSEI',
    'TSX (Canada)': '^GSPTSE',
    'DAX (Germany)': '^GDAXI',
    'FTSE 100 (UK)': '^FTSE',
    'Nikkei 225 (Japan)': '^N225',
    'IPC (Mexico)': '^MXX',
    'Ibovespa (Brazil)': '^BVSP',
}
ytd = {}
for name, tk in indexes.items():
    h = yf.Ticker(tk).history(start='2026-01-01', end='2026-08-21')
    ytd[name] = h['Close'].iloc[-1] / h['Close'].iloc[0] - 1

s = pd.Series(ytd).sort_values(ascending=False)
print(s)
print("Beating S&P500:", (s > s['S&P500 (US)']).sum())

Nikkei 225 (Japan)     0.277507
TSX (Canada)           0.140575
S&P500 (US)            0.114120
FTSE 100 (UK)          0.080102
DAX (Germany)          0.058832
Ibovespa (Brazil)      0.046020
ASX 200 (Australia)    0.040789
IPC (Mexico)           0.003250
Hang Seng (HK)        -0.024298
Shanghai (China)      -0.029750
Nifty 50 (India)      -0.073230
dtype: float64
Beating S&P500: 2


In [155]:
#Question 3
import numpy as np
import pandas as pd

spx = yf.Ticker("^GSPC").history(period="max")['Close']
running_max = spx.cummax()
drawdown = spx / running_max - 1

# Mark whenever we're AT a new all-time high (drawdown == 0)
is_peak = drawdown == 0
peak_id = is_peak.cumsum()   # increments every time we hit a new ATH

# for each "segment" between consecutive peaks, find the worst drawdown
episode_drawdowns = drawdown.groupby(peak_id).min()

# keep only "significant" corrections
threshold = -0.05
significant = episode_drawdowns[episode_drawdowns <= threshold]

pct = sorted([round(-x*100, 1) for x in significant])
print(pct)
print("Median drawdown %:", -np.median(significant)*100)
print("Count of significant corrections:", len(significant))

[5.0, 5.1, 5.1, 5.1, 5.2, 5.3, 5.3, 5.5, 5.5, 5.6, 5.6, 5.6, 5.7, 5.8, 5.8, 5.8, 5.9, 6.1, 6.2, 6.2, 6.3, 6.3, 6.5, 6.5, 6.6, 6.8, 6.8, 6.9, 7.1, 7.1, 7.4, 7.5, 7.6, 7.7, 7.9, 8.1, 8.2, 8.5, 8.9, 9.1, 9.2, 9.4, 9.4, 9.6, 9.6, 9.6, 9.8, 10.1, 10.2, 10.2, 10.3, 10.6, 10.8, 12.1, 14.0, 14.2, 14.4, 18.9, 19.3, 19.8, 19.9, 21.5, 22.2, 25.4, 27.1, 28.0, 33.5, 33.9, 36.1, 48.2, 49.1, 56.8, 86.2]
Median drawdown %: 8.229048188217769
Count of significant corrections: 73


In [153]:
#Question 4
ticker_obj = yf.Ticker('AMZN')
earnings = ticker_obj.get_earnings_dates(limit=30)
earnings = earnings.dropna(subset=['Reported EPS', 'Surprise(%)'])  # drops the 1 future entry

hist = ticker_obj.history(period='max')
hist.index = hist.index.tz_localize(None)

rows = []
for dt, row in earnings.iterrows():
    d = dt.tz_localize(None) if dt.tzinfo else dt
    idx = hist.index.searchsorted(d)
    if 0 < idx < len(hist) - 1:
        day1 = hist['Close'].iloc[idx - 1]
        day3 = hist['Close'].iloc[idx + 1]
        ret_2d = (day3 / day1 - 1) * 100
        rows.append((dt, row['Surprise(%)'], ret_2d))

df = pd.DataFrame(rows, columns=['date', 'surprise', 'ret_2d'])
positive = df[df['surprise'] > 0]
print("Median 2-day return (positive surprises):", positive['ret_2d'].median())
print("Correlation:", df[['surprise', 'ret_2d']].corr())

Median 2-day return (positive surprises): 1.9754103987247307
Correlation:           surprise    ret_2d
surprise  1.000000  0.319284
ret_2d    0.319284  1.000000
